# Problem Set 2

In [30]:
import polars as pl
import numpy as np
import seaborn as sns

# Replication

There is conversation on campus about changes in funding to the Campus Student Learning Center [link](https://www.dailycal.org/news/campus/uc-berkeley-student-learning-center-faces-20-cut-in-peer-tutoring-budget/article_c7b88536-8256-4bd2-ab84-6673986dd06c.html). One of the positive aspects that the learning center provides is support for students classes; another positive aspect is that it creates a sense of belonging and... you got it... social connection and community identity. Or so I would like to believe. But, let's look at the data! 

Recently, Dekker Luberti and Stam examined whether peer-led supplimental instruction (very similar in form to what is offered at the SLC) causes increases in student's grades, but also attitudes associated with wellbeing. [Paper link](https://www.sciencedirect.com/science/article/pii/S0959475223000749). Lets engage with this paper! 

## Experiment Design 

1. Write a short paragraph that describes the core manipulation employed by the authors. Why do they think that this manipulation might effect outcomes they are going to measure? 


Dekker et al. randomly assigned first-year students to either Supplemental Instruction (treatment) or a wait list (control). Students in the treatment group were invited to join seven weeks of 100-minute, peer-led sessions for a course. Then, students in the control group were told they couldn't attend those sessions but could the next quarter. The MANIPULATION lies in the offer of Supplemental Instruction, not the attendance of it. The authors believe Supplemental Instruction could affect the outcome they measure in the study in three ways.  First, Supplemental Instruction gives students more time each week on collaborative learning tasks which should help increase grades. Second, it helps students make friends which can support a sense of belonging. Lastly, working toward shared goals helps build supportive relationships which is assumed to improve mental well-being.


2. Write a few short paragraphs that describe the three outcome *concepts* the authors are interested in, and then how they choose to measure those outcomes. 

Dekker et al. are interested in three outcomes: academic performance (like grades), mental well-being, and a sense of belonging. Academic Performance is judged on how well students can learn the materials in the given course. So the authors measure each student's exam grade on a scale of 0 to 10. The university reports these grades; a minimum of 5.5 is needed to pass the course.
​
For mental well-being, this is a combined measure of subjective and psychological well-being. To measure mental well-being, the authors use the 5 point Warwick-Edinburgh Mental Well-Being Scale. Students rate how often they feel relaxed or connected to others.
​
Lastly, the sense of belonging measures whether a student feels included in the college community. This is measured on 5 point  from Meeuwisse. There are questions like " I feel at home and then they would rate it based on never, or all the time. The last two (well-being and belonging) were both measured with surveys at the start of the quarter(to establish the baseline) and at the end, before exams.


2. Write a short paragraph that identifies other concepts that the authors *could have* be interested in, but did not.


I think the authors could have been interested in other more common outcomes that  Supplemental Instruction is expected to affect. For example, like passing a course, or retention in a class. In my experience as a grad student  tutor these are typically the more traditional ways Supplemental Instruction had been judged.  Also it could have been interesting to measure more specifically into well being. So instead of looking at measuring well-being from a general capacity focusing on things like anxiety or stress maybe through some type of likert scales  before and after it.

2.Then, critically evaluate (this doesn't mean your conclusions need to be negative about it; but rather that you're thinking critically about it) whether the ways the authors have chosen to measure outcomes are likely to capture the concepts they are measuring. 

I think that the author's measure captures the general gist of the three concepts, but it feels a tad bit lopsided.  The grade measure fits academic performance well. Since it’s not self-reported, that removes concerns about students possibly misreporting it.   But I kind of think that since the Supplemental Instruction is built around the lectures,  This could be an issue because basing the measurement on exam success may reflect mastery of lecture or course content rather than overall learning improvement. Also, in general, exams don't truly reflect the amount of work or account for individuals with learning disabilities.
​
The paper stated that the well-being scale is moderately reliable at .75. It covered feeling good and functioning well, which reflects how the authors defined the concept. It falls short because the questions focus on the last two weeks. This feels weak in terms of timing because the Supplemental Instruction spanned many more weeks than the measurement accounts for.Of the three, I would argue Belonging is the weakest because the reliability was only .65. Even the authors admitted that it creates issues with the results. I think it is interesting that part of the measurement was skipping class, which feels more like a behavior??. Also, since Supplemental Instruction is based on a single course, it doesn’t make sense for students to reflect on courses outside of it.


3. Write out the experiment design that the authors employ using the `R, N, X, O, Y` grammar from class. What are the relative strengths of this design, and what are the limitations of this  design? 

For the wellbeing outcomes. Both groups were randomly assigned(R) then they had to take the assessment(O), only the top group got the treatment(X). Then both groups have an outcome.  The only difference is for the grades there isn't a pre test so you wouldn't see an O in the columns.


R     O    X   Y
R     O        Y

for the grades: 

R          X   Y
R              Y


The strength of this study is the random assignment. This removes potential issues like self selection bias. Also the fact that the grades are reported form the university adds a form of authenticity because they aren’t self reported. There's also an ethical nature as the control group also got to experience the support. 

The weakness of this study is the fact that 29% of the treatment never even went to the SI. So then the effects of it can be understated since eit was missing more than 20 % of the treatment group. Also the survey percentages were pretty low at 65% which affects overall study.



# Randomization Inference 

The data for the paper is available at the Open Science Foundation's repository. I have downloaded it, and stored in this repository (bad practice...) in a folder called `./data/raw/`. I have then cleaned up the data for you, so that you don't have to do all the work to clean it yourself. 

Use this data, and randomization inference, to re-analyze the main findings from the paper: 

1. Students who received SI received higher grades; but, 
2. They did not score differently on well-being; or,
3. Belonging scales. 

Please show your code and results for this re-analysis in your homework submission, and include supporting text and interpretation where appropriate to describe and interpret your statistical results. 

In the published analysis, the authors adjust for "demographic covariates". In this reanalysis, you dod not need to make this adjustment. Instead, you can simply model outcomes and treatment.

In [31]:
#**Answer 1:** Students who received SI received higher grades;


df = pl.read_csv(
    "data/raw/SIdataset.csv",
    null_values=["NA", "N/A", "NULL", "null", ""],
    infer_schema_length=None,
    
)


In [ ]:

#**Answer 1:** Students who received SI received higher grades;

## Randomization inference



def estimate_ate(data: pl.DataFrame, condition: str, outcome: str) -> float:
    # average outcome in treatment minus average outcome in control
    return (
        data.filter(pl.col(condition) == 1)[outcome].mean() -
        data.filter(pl.col(condition) == 0)[outcome].mean()
    )

def ri(data: pl.DataFrame, condition: str, outcome: str) -> float:
    shuffle_data = data.with_columns(pl.col(condition).shuffle())
    ri_ate = estimate_ate(shuffle_data, condition, outcome)
    return ri_ate

def p_value(ate: float, ri_vector: list) -> float:
    p_value = (np.abs(ri_vector) >= np.abs(ate)).mean()
    return p_value


#For the grades - answer 1

# we want to keep only students who have a grade
grades = df.select(["Treatment", "Grade"]).drop_nulls()

# the gap
ate = estimate_ate(grades, "Treatment", "Grade")
ris = [ri(grades, "Treatment", "Grade") for _ in range(10000)]

# p value
print("Observed difference in means:", round(ate, 4))
print("P-value from randomization inference:", round(p_value(ate, ris), 4))

Observed difference in means: 0.3673
P-value from randomization inference: 0.0483


**Interpretation For Answer 1*

So from the observed differnce in means we can see that the sutdents in SI(treatment) scored around .3673 higher then then students who were in the control group. To test wether this is by the luck of the universe, Randomization inference was put into play. The people who were in the treatment group were mixed up 10,000 times then check how big the random gap were in grades in comparison to the realone. The P value from this came out to around(approc) .0476 which means only about 4.9%. So it means that aorund 5% of the random gaps were are large as the real one. and the evidence point to the fact that suggest SI helped. But since its pretty close to the .05 , I woulnt argue for it being SUPER strong results. 



In [33]:
#**Answer 2:** 2. They did not score differently on well-being; or


## Randomization inference
# we are only keeping  students who answered the end-of-quarter well-being survey
wellbeing_data = df.select(["Treatment", "T1_Wellbeing"]).drop_nulls()

#  real gap and shuffled gaps
ate_wellbeing = estimate_ate(wellbeing_data, "Treatment", "T1_Wellbeing")
ris_wellbeing = [ri(wellbeing_data, "Treatment", "T1_Wellbeing") for _ in range(10000)]

# p value
print("Observed difference in means:", round(ate_wellbeing, 4))
print("P-value from randomization inference:", round(p_value(ate_wellbeing, ris_wellbeing), 4))



Observed difference in means: -0.0479
P-value from randomization inference: 0.4422


**Interpretation For Answer 2**

So if we look at the observed difference in means: -0.0479.  This means thats between the treament(people in SI) scored around .048 lower then the poeple on the control group(non SI). This is like REALY small gap conserideing this is on a 5 point scale. From here randomization infernce was applied. The value above shows the p value came out to around .4332 which means about 43% of the random gaps were as large as the real one. This supports what the paper found, and i cant really say that SI chnaged wellbeing. but it also doesn't prove that SI has no effect, but that this test can't tell this effect apart.



In [34]:
#Answer 3  They did not score differently on belonging 

belonging_data = df.select(["Treatment", "T1_Belong"]).drop_nulls()


#  real gap and shuffled gaps
ate_belonging = estimate_ate(belonging_data, "Treatment", "T1_Belong")
ris_belonging = [ri(belonging_data, "Treatment", "T1_Belong") for _ in range(10000)]

# p value
print("Observed difference in means:", round(ate_belonging, 4))
print("P-value from randomization inference:", round(p_value(ate_belonging, ris_belonging), 4))


Observed difference in means: 0.0363
P-value from randomization inference: 0.6035


**Interpretation FOR Quesiton 3:**

If we look toward the observed differnce in means we see .0363, whcih means the treatment group(people in SI) score about .036 higher then the control(non-SI) on belonging. Once again this small gap conserding it is on 5 point scale. When randomixzaiton was applied the p vaule came out to around .5931. Which means about 59% of the random gaps were. as large as the rela one. So a gap like this happend alot . And I can't confidently say SI changed belonging. This matches the paper, because it also doesn't prove SI has no effect.


# Power Analysis 

You are intersted in getting people together for a pizza night and you're trying to get off of social media. This leaves you with a conundrum -- how will you get people to join up? The School (don't worry if you're not at the School of Information, suppose that this applies to your department too) always sends an email with informaation that the event is happening, but we get a lot of emails and there might be additional ways to communicate about this event. After a good think, you arrive at two possibilites: 

1. Sending a message through a Canvas announcement to all the students who are enrolled in classes with you; and, 
2. Putting flyers up in the hallways of the building. 

Both of these are a method of broadcast communication, both signal some form of close connection, and both *could* work. Since you're not sure which is going to be effective, suppose that you're going to conduct an experiment to evaluate whether one or another is more effective. Your research design is then, put some people in control; put some people in a condition that sends class updates via bCourses; put some people in a condition that has flyers in the hallways. For now, suppose that you can individually random assign (a characteristic that is almost certainly not actually possible in this secnario). 

Suppose that you know from past events that turnout is about 40%. And, suppose that there are a maximum of roughly 2,000 people that you can outreach (the School has a lot of alumni that also love pizza). Since you do not know how effective the supplimental messages will be, you want to minimize people's exposure to them, if possible. 

Conduct a power analysis to answer the following questions: 

1. How should assignments to treatment be allocated if you are interested in knowing whether **any** supplimental communication is effective? That is, receiving either additional message (1) or (2) from above compared to control? 
2. How should assignments to treatment be allocated if you are intersted in knowing whether a specific communication is effective compared to control? 
3. How should assignments to treatment be allocated if you are interested in knowing whether message (1) or (2) is more effective? 

Because you are being asked to conduct these three different analyses, it seems, *to me* like a good idea to write a function that will conduct a power analysis given inputs that capture assumptions about the world. In answering each of the questions above, produce a plot (or plots) that show participants on the x-axis, power on the y-axis, and possible effect sizes as a color feature for points/lines in the plot.

In [46]:
# Power analysis function

# make one fake datase
def make_data(control_size: int, treatment_size: int, baseline: float, lift: float) -> pl.DataFrame:
    condition = np.repeat([0, 1], 
                          [control_size, treatment_size])
    y = np.random.random(len(condition)) < (baseline + lift * condition)
    # one row per person
    df = pl.DataFrame({
        "id": np.arange(len(condition)),
        "y": y.astype(int),
        "condition": condition,
    })
    return df

# gap in average turnout
def estimate_ate(data: pl.DataFrame, condition: str, outcome: str) -> float:
    return (
        data.filter(pl.col(condition) == 1)[outcome].mean() -
        data.filter(pl.col(condition) == 0)[outcome].mean()
    )

# one random shuffle
def ri(data: pl.DataFrame, condition: str, outcome: str) -> float:
    shuffle_data = data.with_columns(pl.col(condition).shuffle())
    ri_ate = estimate_ate(shuffle_data, condition, outcome)
    return ri_ate

# share of shuffled gaps as big as the real one
def p_value(ate: float, ri_vector: list) -> float:
    p_value = (np.abs(ri_vector) >= np.abs(ate)).mean()
    return p_value

#fake experiment returns a p value
def run_experiment(control_size: int, treatment_size: int, baseline: float, lift: float) -> float:
    df = make_data(control_size, treatment_size, baseline, lift)
    ate = estimate_ate(df, "condition", "y")
    ris = [ri(df, "condition", "y") for _ in range(100)]
    return p_value(ate, ris)

def power(control_size, treatment_size, baseline, lift, number_of_experiments=300):
    p_values_ = [run_experiment(control_size, treatment_size, baseline, lift)
                 for _ in range(number_of_experiments)]
    return np.less_equal(p_values_, 0.05).mean()



**Answer 1:**



**Answer 2:**

_Your answer here._

In [ ]:
# Power analysis question 2: plot here



**Answer 3:**

_Your answer here._